# NYC TLC tip classifier — PySpark + RustFS + MLflow

## What are we building?

A **binary classifier** that predicts, *before* a yellow-taxi trip ends, whether the
passenger will leave a **generous tip** — defined here as a tip worth **≥ 20% of the
fare**. Think of it as the question a dispatcher or a driver-incentive system might
ask: *"given where/when this trip starts and how far it goes, is it likely to be a
high-tip trip?"*

## Where does this fit in the medallion architecture?

This repo is a **medallion lakehouse** (bronze → silver → gold) on top of RustFS, an
S3-compatible object store:

| Layer | Meaning | Here |
| --- | --- | --- |
| **bronze** | raw, as-ingested | the raw NYC TLC parquet files we read below |
| silver | cleaned / conformed | the `clean` dataframe we build in-notebook |
| gold | business-ready / features | the feature vector we feed the model |

We read **bronze** straight from RustFS over the `s3a://` connector and do the
silver/gold shaping live, so you can see every transformation.

## What's the pipeline?

1. **Load** the raw monthly parquet from the bronze bucket (Spark in local mode).
2. **Explore** the data visually (distributions, class balance, tip behaviour by time).
3. **Clean + label** — and carefully avoid two classic traps: *cash-tip bias* and
   *target leakage*.
4. **Engineer features** and assemble a Spark ML pipeline (GBT classifier).
5. **Train and track the experiment in the shared MLflow server** — metrics, plots
   and the model itself are logged centrally, with artifacts living in RustFS.

## What changed vs. the old version?

- **MLflow is now remote.** Earlier this notebook logged to a throwaway local file
  store (`~/mlruns`) that vanished with the pod. Now it logs to the **cluster MLflow
  server**, so every run, metric and model is persisted and comparable across
  notebooks and users.
- **Added exploratory data visualization** so the modelling choices are motivated by
  what the data actually looks like, not by guesswork.

> Credentials for the bronze bucket arrive as injected `AWS_*` env vars (a
> bronze-scoped RustFS key). We never hard-code secrets in the notebook.

## 0. Start Spark

Spark runs in **local mode** (`local[*]`, all cores) inside this single-user notebook
pod — there is no separate Spark cluster. The `s3a` settings below are what let Spark
talk to RustFS instead of real AWS S3:

- `path.style.access = true` — RustFS/MinIO use *path-style* URLs
  (`endpoint/bucket/key`), not AWS's *virtual-host* style (`bucket.endpoint/key`).
- `connection.ssl.enabled = false` — the in-cluster endpoint is plain HTTP.
- `input.stream.type = classic` — Hadoop 3.5's new *analytics* S3 input stream is
  tuned for AWS S3 semantics and misbehaves against RustFS, so we force the classic
  reader.
- `SimpleAWSCredentialsProvider` — take the key/secret straight from the config we
  pass (the injected env vars), no AWS metadata lookups.

In [ ]:
import os
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder.appName('tlc-tip')
    .master('local[*]')
    .config('spark.hadoop.fs.s3a.endpoint', os.environ['AWS_ENDPOINT_URL'])
    .config('spark.hadoop.fs.s3a.access.key', os.environ['AWS_ACCESS_KEY_ID'])
    .config('spark.hadoop.fs.s3a.secret.key', os.environ['AWS_SECRET_ACCESS_KEY'])
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')        # RustFS/MinIO
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')  # endpoint is http
    .config('spark.hadoop.fs.s3a.input.stream.type', 'classic')     # Hadoop 3.5 analytics stream is for AWS S3, not RustFS
    .config('spark.hadoop.fs.s3a.aws.credentials.provider',
            'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider')
    .config('spark.driver.memory', '4g')
    .getOrCreate())
spark

## 1. Load the raw bronze data

We read one month of **NYC TLC yellow-taxi** trips. Each row is one trip; the columns
we care about are:

| Column | Meaning |
| --- | --- |
| `tpep_pickup_datetime` / `tpep_dropoff_datetime` | trip start / end timestamps |
| `trip_distance` | miles travelled |
| `fare_amount` | metered fare (**before** tip/tolls/taxes) |
| `tip_amount` | tip recorded by the meter |
| `total_amount` | grand total (**includes** the tip — danger, see below) |
| `payment_type` | 1 = card, 2 = cash, ... |
| `PULocationID` / `DOLocationID` | pickup / drop-off TLC zone IDs |

Parquet is columnar, so `count()` and schema reads are cheap even over S3.

In [ ]:
bucket = os.environ.get('BRONZE_BUCKET', 'bronze')
key = 'yellow_tripdata/yellow_tripdata_2024-01.parquet'
df = spark.read.parquet(f's3a://{bucket}/{key}')
print('rows:', df.count())
df.printSchema()

## 2. Exploratory data analysis (EDA)

Before modelling, **look at the data**. EDA tells us the shape of the distributions,
the amount of garbage (negative fares, 0-mile trips), and — critically for this
problem — how tips behave.

Spark can hold tens of millions of rows, so we never pull the whole frame into
pandas. Instead we take a small **random sample** (2%) just for plotting; aggregate
statistics that need the full data are computed in Spark and only the tiny result is
collected.

In [ ]:
import matplotlib.pyplot as plt

sample_cols = ['fare_amount', 'trip_distance', 'tip_amount', 'payment_type',
               'passenger_count']
pdf = (df.select(*sample_cols)
         .sample(fraction=0.02, seed=42)
         .toPandas())

# Tip as a share of fare (guard against divide-by-zero / negative fares).
pdf = pdf[pdf['fare_amount'] > 0].copy()
pdf['tip_pct'] = pdf['tip_amount'] / pdf['fare_amount']
print(f'EDA sample rows: {len(pdf):,}')
pdf.head()

In [ ]:
# Four quick views of the raw data. We clip extreme values only for the plots
# (not for the model) so a few outliers don't flatten the histograms.
fig_eda, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0, 0].hist(pdf['fare_amount'].clip(0, 80), bins=40, color='#4c72b0')
axes[0, 0].set_title('Fare amount ($)')
axes[0, 0].set_xlabel('fare'); axes[0, 0].set_ylabel('trips')

axes[0, 1].hist(pdf['trip_distance'].clip(0, 20), bins=40, color='#55a868')
axes[0, 1].set_title('Trip distance (miles)')
axes[0, 1].set_xlabel('miles')

axes[1, 0].hist(pdf['tip_pct'].clip(0, 0.6), bins=40, color='#c44e52')
axes[1, 0].axvline(0.20, color='black', ls='--', label='20% label threshold')
axes[1, 0].set_title('Tip as % of fare')
axes[1, 0].set_xlabel('tip / fare'); axes[1, 0].legend()

pay = pdf['payment_type'].value_counts().sort_index()
axes[1, 1].bar(pay.index.astype(str), pay.values, color='#8172b3')
axes[1, 1].set_title('Payment type (1=card, 2=cash)')
axes[1, 1].set_xlabel('payment_type'); axes[1, 1].set_ylabel('trips')

fig_eda.suptitle('NYC TLC yellow taxi — raw bronze sample', fontsize=14)
fig_eda.tight_layout()
plt.show()

**What the plots tell us (and why it shapes the model):**

- The **tip-% histogram** usually shows a big spike at exactly `0`. That is almost
  entirely **cash trips**: cash tips are handed over physically and never reach the
  meter, so they are recorded as `0`. If we trained on those, the model would learn
  "cash → no tip", which is a **recording artefact, not reality**. → We keep only
  **card** payments (`payment_type = 1`).
- Fares and distances have a **long right tail** and some non-positive junk values.
  → We filter to plausible ranges when cleaning.
- The dashed line marks our **20% label threshold**; everything to its right is a
  "generous tip" (label = 1).

## 3. Clean + label

Two decisions matter more than anything else here:

1. **Cash-tip bias** → keep only card payments (`payment_type = 1`).
2. **Target leakage** → never feed the model `total_amount` (or `tip_amount`), because
   they already *contain* the answer. A feature is only legitimate if it is knowable
   *before* the tip happens. Using `total_amount` would give a fake ~perfect score
   that collapses in production.

We also drop obviously dirty rows (non-positive fares, implausible distances or
durations) and derive a few honest features: trip duration, pickup hour, day of week,
and the pickup/drop-off zones as categoricals.

In [ ]:
clean = (df
    .withColumn('duration_min',
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 60)
    .filter((F.col('payment_type') == 1)                       # card only (see EDA note)
            & (F.col('fare_amount') > 0)                       # drop junk fares
            & (F.col('trip_distance').between(0.1, 100))       # plausible distance
            & (F.col('duration_min').between(1, 180)))         # 1 min .. 3 h
    .withColumn('label', (F.col('tip_amount') / F.col('fare_amount') >= 0.20).cast('double'))
    .withColumn('hour', F.hour('tpep_pickup_datetime'))
    .withColumn('dow',  F.dayofweek('tpep_pickup_datetime'))   # 1 = Sunday
    .withColumn('pu', F.col('PULocationID').cast('string'))
    .withColumn('do', F.col('DOLocationID').cast('string')))

# Class balance: if one class dominates, plain accuracy is misleading and we should
# lean on AUC / AUC-PR instead.
clean.groupBy('label').count().orderBy('label').show()

In [ ]:
# Does tipping behaviour depend on WHEN the trip happens? These aggregations run on
# the full cleaned data in Spark; only the small per-hour / per-day result comes back.
by_hour = (clean.groupBy('hour')
                .agg(F.avg('label').alias('tip_rate'), F.count('*').alias('trips'))
                .orderBy('hour').toPandas())
by_dow = (clean.groupBy('dow')
               .agg(F.avg('label').alias('tip_rate'), F.count('*').alias('trips'))
               .orderBy('dow').toPandas())

dow_names = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']  # Spark dayofweek: 1=Sun
fig_rate, (ax_h, ax_d) = plt.subplots(1, 2, figsize=(12, 4))

ax_h.bar(by_hour['hour'], by_hour['tip_rate'], color='#4c72b0')
ax_h.set_title('Generous-tip rate by pickup hour')
ax_h.set_xlabel('hour of day'); ax_h.set_ylabel('share tipping ≥ 20%')

ax_d.bar([dow_names[d - 1] for d in by_dow['dow']], by_dow['tip_rate'], color='#55a868')
ax_d.set_title('Generous-tip rate by day of week')
ax_d.set_xlabel('day')

fig_rate.tight_layout()
plt.show()

If these bars are roughly flat, time-of-day is a weak signal; if they wiggle, the
model has something real to latch onto. Either way, `hour` and `dow` are **leak-free**
features (known the moment the trip starts), so they are safe to use.

## 4. Feature engineering + model pipeline

A Spark ML **Pipeline** chains the transforms so the exact same steps apply to train,
test and future data:

1. `StringIndexer` — turn the high-cardinality zone strings (`pu`, `do`) into integer
   indices. `handleInvalid='keep'` sends unseen zones to a special bucket instead of
   crashing.
2. `OneHotEncoder` — expand those indices into sparse binary vectors (no false
   ordering between zone IDs).
3. `VectorAssembler` — glue all numeric + encoded features into a single `features`
   vector.
4. `GBTClassifier` — gradient-boosted trees: strong on tabular data and tolerant of
   mixed feature scales.

We split 80/20 with a fixed seed so the run is reproducible.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import GBTClassifier

idx = StringIndexer(inputCols=['pu', 'do'], outputCols=['pu_i', 'do_i'], handleInvalid='keep')
ohe = OneHotEncoder(inputCols=['pu_i', 'do_i'], outputCols=['pu_v', 'do_v'])
vec = VectorAssembler(
    inputCols=['trip_distance', 'duration_min', 'fare_amount', 'hour', 'dow',
               'passenger_count', 'pu_v', 'do_v'],
    outputCol='features', handleInvalid='skip')
gbt = GBTClassifier(labelCol='label', maxIter=50, maxDepth=6)
pipeline = Pipeline(stages=[idx, ohe, vec, gbt])

train, test = clean.randomSplit([0.8, 0.2], seed=42)
print(f'train rows: {train.count():,}   test rows: {test.count():,}')

## 5. Train and track in the shared MLflow server

This is the part that changed. Instead of a local `~/mlruns` folder, we point MLflow
at the **cluster tracking server**.

- The URI comes from the `MLFLOW_TRACKING_URI` env var (injected by JupyterHub), with
  a fallback to the in-cluster service DNS. Nothing is hard-coded.
- The server runs in **proxied-artifact mode**: when we call `log_figure` / `log_model`,
  the artifacts are streamed *through* the MLflow server, which stores them in the
  `mlflow` bucket on **RustFS**. The notebook only holds a bronze-scoped key and never
  needs the artifact-bucket credentials — that is the whole point of proxied mode.

In [ ]:
import mlflow

# Centralised config via env, with an in-cluster default so the notebook also works
# if the env var is ever missing.
MLFLOW_URI = os.environ.get('MLFLOW_TRACKING_URI', 'http://mlflow.mlflow.svc.cluster.local')
mlflow.set_tracking_uri(MLFLOW_URI)
print('MLflow tracking server:', mlflow.get_tracking_uri())

In [ ]:
import mlflow.spark
import numpy as np
from pyspark.ml.functions import vector_to_array
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import roc_curve, confusion_matrix

mlflow.set_experiment('tlc-tip-classifier')

with mlflow.start_run(run_name='gbt-2024-01') as run:
    mlflow.log_params({
        'algo': 'GBTClassifier', 'maxIter': 50, 'maxDepth': 6,
        'data_month': '2024-01', 'payment_filter': 'card-only',
        'label_rule': 'tip/fare >= 0.20',
    })

    model = pipeline.fit(train)
    pred = model.transform(test)

    auc_roc = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderROC').evaluate(pred)
    auc_pr = BinaryClassificationEvaluator(labelCol='label', metricName='areaUnderPR').evaluate(pred)
    mlflow.log_metric('auc_roc', auc_roc)
    mlflow.log_metric('auc_pr', auc_pr)
    print(f'AUC-ROC: {auc_roc:.4f}   AUC-PR: {auc_pr:.4f}')

    # Bring test-set scores to the driver for sklearn-based plots. probability is a
    # Spark vector; vector_to_array(...)[1] is P(label = 1).
    scored = (pred.select('label', vector_to_array('probability')[1].alias('p'), 'prediction')
                  .toPandas())
    y, p, yhat = scored['label'].values, scored['p'].values, scored['prediction'].values

    # ROC curve.
    fpr, tpr, _ = roc_curve(y, p)
    fig_roc, ax_roc = plt.subplots(figsize=(5, 5))
    ax_roc.plot(fpr, tpr, color='#4c72b0', label=f'AUC = {auc_roc:.3f}')
    ax_roc.plot([0, 1], [0, 1], '--', color='gray')
    ax_roc.set_xlabel('false positive rate'); ax_roc.set_ylabel('true positive rate')
    ax_roc.set_title('ROC curve'); ax_roc.legend(loc='lower right')
    fig_roc.tight_layout()

    # Confusion matrix at the default 0.5 threshold.
    cm = confusion_matrix(y, yhat)
    fig_cm, ax_cm = plt.subplots(figsize=(4.5, 4.5))
    ax_cm.imshow(cm, cmap='Blues')
    for (i, j), v in np.ndenumerate(cm):
        ax_cm.text(j, i, f'{v:,}', ha='center', va='center')
    ax_cm.set_xticks([0, 1]); ax_cm.set_yticks([0, 1])
    ax_cm.set_xticklabels(['not generous', 'generous'])
    ax_cm.set_yticklabels(['not generous', 'generous'])
    ax_cm.set_xlabel('predicted'); ax_cm.set_ylabel('actual')
    ax_cm.set_title('Confusion matrix (thr = 0.5)')
    fig_cm.tight_layout()

    # Log every figure as an artifact -> stored in RustFS via the proxied store.
    mlflow.log_figure(fig_eda,  'eda/raw_distributions.png')
    mlflow.log_figure(fig_rate, 'eda/tip_rate_by_time.png')
    mlflow.log_figure(fig_roc,  'eval/roc_curve.png')
    mlflow.log_figure(fig_cm,   'eval/confusion_matrix.png')

    # Log the fitted Spark pipeline model (also uploaded to RustFS via the server).
    mlflow.spark.log_model(model, 'model')

    mlflow.set_tags({'framework': 'pyspark', 'stage': 'experiment', 'dataset': 'nyc-tlc-yellow'})
    print('run_id:', run.info.run_id)
    print('artifacts:', mlflow.get_artifact_uri())

## 6. See it in the MLflow UI

Open the MLflow server (its LoadBalancer IP on the LAN, e.g. `http://192.168.1.243`)
and look under the **`tlc-tip-classifier`** experiment. For the run you just created
you'll find:

- **Parameters & metrics** — `auc_roc`, `auc_pr`, the hyper-parameters.
- **Artifacts** — the four PNGs (`eda/…`, `eval/…`) and the serialized Spark model,
  all physically stored in the `mlflow` bucket on RustFS.
- **Compare** — run this notebook again with different `maxDepth`/`maxIter` or a
  different month and MLflow will line the runs up side by side.

### Next steps
- **Class imbalance:** if the labels are skewed, prefer `auc_pr` and consider a
  `weightCol` on the classifier.
- **Realistic evaluation:** swap `randomSplit` for a **temporal split** (train on
  Jan–Apr, test on May–Jun) so you measure performance the way production sees it.
- **Schema drift:** when reading several months, read each file, cast to a common
  schema, then `unionByName` before training.
- **Model registry:** promote a good run with `mlflow.register_model(...)` to version
  and stage it for serving.